# 03 — Modeling, Evaluation & SHAP Analysis
Notebook ini membangun dan mengevaluasi semua model:
- Training 3 base models
- Soft Voting Ensemble
- ROC Curve comparison
- SHAP explainability (summary + waterfall)
- Learning curve analysis

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
from sklearn.metrics import roc_curve, auc as sk_auc
from sklearn.model_selection import learning_curve, StratifiedKFold
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

from preprocessing import (
    load_dataset, feature_engineering,
    build_preprocessor, prepare_data, get_feature_names_out
)
from models import (
    build_ensemble, build_random_forest, build_xgboost,
    build_logistic_regression, build_full_pipeline,
    evaluate_model, compare_models, save_pipeline
)
from explainer import (
    get_shap_values, plot_summary, plot_bar_importance, plot_waterfall
)
import shap
print('Import berhasil ✓')

## 1. Persiapan Data

In [ ]:
df = load_dataset('../data/heart.csv')
df = feature_engineering(df)
X_train, X_test, y_train, y_test = prepare_data(df, apply_smote=True)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 2. Training Semua Model

In [ ]:
results = []

# ── Random Forest ──
rf_pipeline = build_full_pipeline(build_preprocessor(), build_random_forest())
rf_result = evaluate_model(rf_pipeline, X_train, y_train, X_test, y_test,
                            model_name='Random Forest')
results.append(rf_result)

# ── XGBoost ──
xgb_pipeline = build_full_pipeline(build_preprocessor(), build_xgboost())
xgb_result = evaluate_model(xgb_pipeline, X_train, y_train, X_test, y_test,
                             model_name='XGBoost')
results.append(xgb_result)

# ── Logistic Regression ──
lr_pipeline = build_full_pipeline(build_preprocessor(), build_logistic_regression())
lr_result = evaluate_model(lr_pipeline, X_train, y_train, X_test, y_test,
                            model_name='Logistic Regression')
results.append(lr_result)

# ── Soft Voting Ensemble ──
ensemble_pipeline = build_full_pipeline(build_preprocessor(), build_ensemble())
ensemble_result = evaluate_model(ensemble_pipeline, X_train, y_train, X_test, y_test,
                                  model_name='Soft Voting Ensemble')
results.append(ensemble_result)

## 3. Tabel Perbandingan Model

In [ ]:
comparison_df = compare_models(results)
comparison_df.to_csv('../reports/model_comparison.csv')
print('Tersimpan ke ../reports/model_comparison.csv ✓')

In [ ]:
# Visualisasi perbandingan
metrics = ['accuracy', 'recall', 'f1', 'auc_roc']
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(comparison_df))
width = 0.2
colors = ['#1D9E75', '#534AB7', '#D85A30', '#BA7517']

for i, metric in enumerate(metrics):
    bars = ax.bar(x + i*width, comparison_df[metric], width,
                  label=metric.upper().replace('_','-'),
                  color=colors[i], alpha=0.85, edgecolor='white')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(comparison_df.index, rotation=12, ha='right')
ax.set_ylim(0.70, 1.02)
ax.set_ylabel('Score')
ax.set_title('Perbandingan Metrik — Semua Model', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../reports/figures/03_model_comparison.png', bbox_inches='tight')
plt.show()

## 4. ROC Curve — Semua Model

In [ ]:
pipelines = {
    'Random Forest':       rf_pipeline,
    'XGBoost':             xgb_pipeline,
    'Logistic Regression': lr_pipeline,
    'Soft Voting Ensemble': ensemble_pipeline,
}
colors_roc = ['#2E86AB', '#E84855', '#F5A623', '#1D9E75']

fig, ax = plt.subplots(figsize=(8, 6))

for (name, pipe), color in zip(pipelines.items(), colors_roc):
    y_proba = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = sk_auc(fpr, tpr)
    lw = 2.5 if 'Ensemble' in name else 1.5
    ls = '-' if 'Ensemble' in name else '--'
    ax.plot(fpr, tpr, color=color, linewidth=lw, linestyle=ls,
            label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5, label='Random (AUC = 0.500)')
ax.set_xlim([0,1])
ax.set_ylim([0,1.02])
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curve — Perbandingan Semua Model', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../reports/figures/03_roc_curves.png', bbox_inches='tight')
plt.show()

## 5. Learning Curve — Ensemble

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    build_full_pipeline(build_preprocessor(), build_ensemble()),
    X_train, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    train_sizes=np.linspace(0.1, 1.0, 8),
    scoring='roc_auc',
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_mean, 'o-', color='#1D9E75', label='Training AUC', linewidth=2)
ax.fill_between(train_sizes, train_mean-train_std, train_mean+train_std,
                alpha=0.15, color='#1D9E75')
ax.plot(train_sizes, val_mean, 'o-', color='#534AB7', label='Validation AUC', linewidth=2)
ax.fill_between(train_sizes, val_mean-val_std, val_mean+val_std,
                alpha=0.15, color='#534AB7')
ax.set_xlabel('Training Samples', fontsize=11)
ax.set_ylabel('AUC-ROC', fontsize=11)
ax.set_title('Learning Curve — Soft Voting Ensemble', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0.7, 1.01)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../reports/figures/03_learning_curve.png', bbox_inches='tight')
plt.show()

## 6. SHAP Analysis

In [ ]:
# Preprocess test data untuk SHAP
preprocessor_fit = ensemble_pipeline.named_steps['preprocessor']
X_test_proc = preprocessor_fit.transform(X_test)
feature_names = get_feature_names_out(preprocessor_fit)
print(f'X_test preprocessed: {X_test_proc.shape}')

In [ ]:
# Hitung SHAP values menggunakan XGBoost (TreeExplainer — paling cepat)
explainer, shap_values = get_shap_values(
    ensemble_pipeline, X_test_proc, model_name='xgboost'
)
print(f'SHAP values shape: {shap_values.shape}')

In [ ]:
# SHAP Summary Plot (Beeswarm)
fig = plot_summary(
    shap_values, X_test_proc, feature_names,
    save_path='../reports/figures/03_shap_summary.png'
)
plt.show()

In [ ]:
# Bar Importance
fig = plot_bar_importance(
    shap_values, feature_names,
    save_path='../reports/figures/03_shap_importance.png'
)
plt.show()

In [ ]:
# Waterfall untuk 3 pasien berbeda (low risk, high risk, borderline)
y_pred_proba = ensemble_pipeline.predict_proba(X_test)[:, 1]

idx_high   = np.where(y_pred_proba > 0.8)[0][0]
idx_low    = np.where(y_pred_proba < 0.2)[0][0]
idx_border = np.argmin(np.abs(y_pred_proba - 0.5))

for idx, label in [(idx_high,'high'), (idx_low,'low'), (idx_border,'borderline')]:
    proba = y_pred_proba[idx]
    print(f'Pasien [{label}] — Probabilitas: {proba:.3f}')
    fig = plot_waterfall(
        explainer, shap_values, X_test_proc, feature_names,
        patient_idx=idx,
        save_path=f'../reports/figures/03_shap_waterfall_{label}.png'
    )
    plt.show()
    plt.close()

## 7. Simpan Model Final

In [ ]:
import os
os.makedirs('../models', exist_ok=True)
save_pipeline(ensemble_pipeline, '../models/ensemble_pipeline.joblib')
save_pipeline(rf_pipeline,       '../models/rf_pipeline.joblib')
save_pipeline(xgb_pipeline,      '../models/xgb_pipeline.joblib')
print('Semua model tersimpan ✓')

## 8. Ringkasan Akhir

In [ ]:
best = comparison_df['auc_roc'].idxmax()
print('═'*55)
print('RINGKASAN HASIL MODELING')
print('═'*55)
print()
print('Perbandingan model (diurutkan berdasarkan AUC-ROC):')
print(comparison_df[['accuracy','recall','f1','auc_roc']].round(4).to_string())
print()
print(f'Model terbaik  : {best}')
print(f'AUC-ROC        : {comparison_df.loc[best,"auc_roc"]:.4f}')
print(f'Recall         : {comparison_df.loc[best,"recall"]:.4f}')
print()
print('File yang dihasilkan:')
print('  models/ensemble_pipeline.joblib')
print('  models/rf_pipeline.joblib')
print('  models/xgb_pipeline.joblib')
print('  reports/model_comparison.csv')
print('  reports/figures/*.png')
print()
print('Jalankan Streamlit app:')
print('  streamlit run app/streamlit_app.py')